# Birmingham External Test Preparation
## Atlanta-trained LSTM → Birmingham Cross-City Evaluation

This notebook prepares **Birmingham as a completely external test city**.

### Experimental rule

Atlanta is the only city used for model development:

```text
Atlanta
  ├── 70% Training
  ├── 15% Validation
  └── 15% Internal Test
```

Birmingham is used only after the Atlanta model has been trained and selected:

```text
Birmingham
  └── 100% External Test
```

### Critical preprocessing rule

The Birmingham test data must use:

- Atlanta's feature definitions
- Atlanta's fitted `X_scaler`
- Atlanta's fitted `y_scaler`

**No Birmingham scaler is fitted anywhere in this notebook.**

This ensures that the Birmingham result is a genuine cross-city external evaluation rather than a separately normalized experiment.

### Output

This notebook creates:

```text
birmingham_external_X_test.npy
birmingham_external_y_test.npy
birmingham_external_metadata.csv
```

These are the files loaded by the LSTM training notebook.


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Libraries loaded.")


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Change this folder if your files are stored elsewhere.
DATA_DIR = "../cleaning"

# Raw Birmingham master CSV
BIRMINGHAM_RAW_FILE = os.path.join(
    DATA_DIR,
    "birmingham_master.csv"
)

# Atlanta scalers
ATLANTA_X_SCALER_FILE = os.path.join(
    DATA_DIR,
    "atlanta_X_scaler.pkl"
)

ATLANTA_Y_SCALER_FILE = os.path.join(
    DATA_DIR,
    "atlanta_y_scaler.pkl"
)

# External-test outputs
OUTPUT_X = os.path.join(
    DATA_DIR,
    "birmingham_external_X_test.npy"
)

OUTPUT_Y = os.path.join(
    DATA_DIR,
    "birmingham_external_y_test.npy"
)

OUTPUT_METADATA = os.path.join(
    DATA_DIR,
    "birmingham_external_metadata.csv"
)

SEQUENCE_LENGTH = 24

print("Birmingham input:", BIRMINGHAM_RAW_FILE)
print("Atlanta X scaler:", ATLANTA_X_SCALER_FILE)
print("Atlanta y scaler:", ATLANTA_Y_SCALER_FILE)


In [ ]:
# ============================================================
# 3. CHECK FILES
# ============================================================

required_files = [
    BIRMINGHAM_RAW_FILE,
    ATLANTA_X_SCALER_FILE,
    ATLANTA_Y_SCALER_FILE
]

missing = [
    path for path in required_files
    if not os.path.exists(path)
]

if missing:
    raise FileNotFoundError(
        "The following required files were not found:\n\n"
        + "\n".join(missing)
    )

print("All required input files exist.")


In [ ]:
# ============================================================
# 4. LOAD BIRMINGHAM RAW DATA
# ============================================================

df = pd.read_csv(BIRMINGHAM_RAW_FILE)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


In [ ]:
# ============================================================
# 5. VALIDATE REQUIRED COLUMNS
# ============================================================

TARGET = "hourly_load_mw"

required_columns = [
    "timestamp",
    "hourly_load_mw",
    "temperature_c",
    "dew_point_c",
    "relative_humidity_percent",
    "station_pressure_kpa",
    "wind_direction_deg",
    "wind_speed_ms",
    "horizontal_visibility_km",
    "hourly_precipitation_cm"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required Birmingham columns:\n"
        + "\n".join(missing_columns)
    )

print("All required Birmingham columns are present.")


In [ ]:
# ============================================================
# 6. TIMESTAMP PREPARATION
# ============================================================

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df = df.sort_values("timestamp").reset_index(drop=True)

if df["timestamp"].isna().any():
    raise ValueError("Birmingham contains invalid timestamps.")

print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())


In [ ]:
# ============================================================
# 7. REMOVE DUPLICATE TIMESTAMPS
# ============================================================

duplicate_timestamps = df["timestamp"].duplicated().sum()

print("Duplicate timestamps:", duplicate_timestamps)

df = (
    df
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Rows after duplicate removal:", len(df))


In [ ]:
# ============================================================
# 8. CREATE THE SAME CALENDAR FEATURES USED FOR ATLANTA
# ============================================================

# IMPORTANT:
# Do not use Birmingham's existing 1-24 hour column.
# Recreate the model hour consistently from timestamp as 0-23.

df["hour_lstm"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_of_month"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)


In [ ]:
# ============================================================
# 9. TIME OF DAY
# ============================================================

def get_time_period(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    elif 18 <= hour < 22:
        return "Evening"
    else:
        return "Night"


df["time_period"] = df["hour_lstm"].apply(get_time_period)

df = pd.get_dummies(
    df,
    columns=["time_period"],
    dtype=int
)

time_columns = [
    "time_period_Morning",
    "time_period_Afternoon",
    "time_period_Evening",
    "time_period_Night"
]

for col in time_columns:
    if col not in df.columns:
        df[col] = 0


In [ ]:
# ============================================================
# 10. SEASON
# ============================================================

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


df["season"] = df["month"].apply(get_season)

df = pd.get_dummies(
    df,
    columns=["season"],
    dtype=int
)

season_columns = [
    "season_Winter",
    "season_Spring",
    "season_Summer",
    "season_Autumn"
]

for col in season_columns:
    if col not in df.columns:
        df[col] = 0


In [ ]:
# ============================================================
# 11. WEATHER ENGINEERING
# ============================================================

df["temperature_dewpoint_diff"] = (
    df["temperature_c"] - df["dew_point_c"]
)

df["temperature_squared"] = (
    df["temperature_c"] ** 2
)

df["humidity_squared"] = (
    df["relative_humidity_percent"] ** 2
)

df["temperature_humidity_interaction"] = (
    df["temperature_c"]
    * df["relative_humidity_percent"]
)


In [ ]:
# ============================================================
# 12. PRIMARY FEATURE LIST
# ============================================================

# This must match the Atlanta training feature definition.

feature_columns = [
    # Weather
    "temperature_c",
    "dew_point_c",
    "relative_humidity_percent",
    "station_pressure_kpa",
    "wind_direction_deg",
    "wind_speed_ms",
    "horizontal_visibility_km",
    "hourly_precipitation_cm",

    # Calendar
    "hour_lstm",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend",

    # Time of day
    "time_period_Morning",
    "time_period_Afternoon",
    "time_period_Evening",
    "time_period_Night",

    # Season
    "season_Winter",
    "season_Spring",
    "season_Summer",
    "season_Autumn",

    # Engineered weather
    "temperature_dewpoint_diff",
    "temperature_squared",
    "humidity_squared",
    "temperature_humidity_interaction"
]

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        "The following model features are missing from Birmingham:\n"
        + "\n".join(missing_features)
    )

print("Number of model features:", len(feature_columns))

for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")


In [ ]:
# ============================================================
# 13. LOAD ATLANTA SCALERS
# ============================================================

X_scaler = joblib.load(ATLANTA_X_SCALER_FILE)
y_scaler = joblib.load(ATLANTA_Y_SCALER_FILE)

print("Atlanta X scaler loaded:", type(X_scaler))
print("Atlanta y scaler loaded:", type(y_scaler))

if not isinstance(X_scaler, MinMaxScaler):
    print("Warning: X scaler is not a MinMaxScaler.")

if not isinstance(y_scaler, MinMaxScaler):
    print("Warning: y scaler is not a MinMaxScaler.")


In [ ]:
# ============================================================
# 14. VERIFY SCALER FEATURE COUNT
# ============================================================

expected_features = len(feature_columns)

if hasattr(X_scaler, "n_features_in_"):
    print(
        "Atlanta scaler expects:",
        X_scaler.n_features_in_,
        "features"
    )

    if X_scaler.n_features_in_ != expected_features:
        raise ValueError(
            "Feature-count mismatch!\n"
            f"Atlanta scaler expects {X_scaler.n_features_in_} features, "
            f"but this notebook created {expected_features}."
        )

print("Scaler feature count is compatible.")


In [ ]:
# ============================================================
# 15. SELECT BIRMINGHAM FEATURES AND TARGET
# ============================================================

X_birmingham = df[feature_columns].copy()
y_birmingham = df[[TARGET]].copy()

print("X:", X_birmingham.shape)
print("y:", y_birmingham.shape)

print("\nMissing X values:", X_birmingham.isna().sum().sum())
print("Missing y values:", y_birmingham.isna().sum().sum())


In [ ]:
# ============================================================
# 16. HANDLE MISSING VALUES
# ============================================================

# For external testing we do NOT fit an imputer on Birmingham.
# Rows with missing model inputs/target are removed.

valid_mask = (
    X_birmingham.notna().all(axis=1)
    & y_birmingham[TARGET].notna()
)

df = df.loc[valid_mask].reset_index(drop=True)
X_birmingham = X_birmingham.loc[valid_mask].reset_index(drop=True)
y_birmingham = y_birmingham.loc[valid_mask].reset_index(drop=True)

print("Rows remaining:", len(df))


In [ ]:
# ============================================================
# 17. APPLY ATLANTA X SCALER
# ============================================================

# NEVER call fit() or fit_transform() here.
# Birmingham is transformed using the already-fitted Atlanta scaler.

X_birmingham_scaled = X_scaler.transform(
    X_birmingham
)

print("Birmingham features transformed using Atlanta X scaler.")
print("Scaled shape:", X_birmingham_scaled.shape)


In [ ]:
# ============================================================
# 18. APPLY ATLANTA y SCALER
# ============================================================

# Again, no fitting occurs here.

y_birmingham_scaled = y_scaler.transform(
    y_birmingham[[TARGET]]
)

print("Birmingham target transformed using Atlanta y scaler.")
print("Scaled target shape:", y_birmingham_scaled.shape)


In [ ]:
# ============================================================
# 19. CREATE 24-HOUR EXTERNAL TEST SEQUENCES
# ============================================================

def create_external_sequences(X, y, timestamps, sequence_length):
    X_sequences = []
    y_sequences = []
    target_timestamps = []

    for i in range(sequence_length, len(X)):
        X_sequences.append(
            X[i - sequence_length:i]
        )

        y_sequences.append(
            y[i]
        )

        target_timestamps.append(
            timestamps.iloc[i]
        )

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_sequences, dtype=np.float32),
        pd.Series(target_timestamps, name="timestamp")
    )


X_external, y_external, external_timestamps = (
    create_external_sequences(
        X_birmingham_scaled,
        y_birmingham_scaled,
        df["timestamp"],
        SEQUENCE_LENGTH
    )
)

print("External X shape:", X_external.shape)
print("External y shape:", y_external.shape)
print("External timestamps:", external_timestamps.shape)


In [ ]:
# ============================================================
# 20. VERIFY FINAL SHAPE
# ============================================================

expected_shape = (
    len(X_external),
    SEQUENCE_LENGTH,
    len(feature_columns)
)

if X_external.shape != expected_shape:
    raise ValueError(
        f"Unexpected X shape: {X_external.shape}; "
        f"expected {expected_shape}"
    )

if y_external.shape != (len(X_external), 1):
    raise ValueError(
        f"Unexpected y shape: {y_external.shape}"
    )

print("Final external-test shape is correct.")


In [ ]:
# ============================================================
# 21. VERIFY NO BIRMINGHAM SCALER WAS FIT
# ============================================================

print("External-test preprocessing check")
print("----------------------------------")
print("X scaler source: Atlanta")
print("y scaler source: Atlanta")
print("Birmingham scaler fitting: NONE")
print("Birmingham train split used: NO")
print("Birmingham validation split used: NO")
print("Birmingham hyperparameter tuning: NO")
print("Birmingham model selection: NO")


In [ ]:
# ============================================================
# 22. SAVE EXTERNAL TEST ARRAYS
# ============================================================

np.save(
    OUTPUT_X,
    X_external
)

np.save(
    OUTPUT_Y,
    y_external
)

print("Saved:")
print(OUTPUT_X)
print(OUTPUT_Y)


In [ ]:
# ============================================================
# 23. SAVE EXTERNAL TEST METADATA
# ============================================================

metadata = pd.DataFrame({
    "timestamp": external_timestamps,
    "actual_load_mw": y_birmingham.loc[
        y_birmingham.index[
            SEQUENCE_LENGTH:
        ]
    ][TARGET].values
})

metadata.to_csv(
    OUTPUT_METADATA,
    index=False
)

print("Saved metadata:")
print(OUTPUT_METADATA)

print("\nMetadata preview:")
print(metadata.head())


In [ ]:
# ============================================================
# 24. FINAL SANITY CHECK
# ============================================================

print("=" * 70)
print("BIRMINGHAM EXTERNAL TEST PREPARATION COMPLETE")
print("=" * 70)

print("\nRaw Birmingham rows:", len(df))
print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", len(feature_columns))
print("External test samples:", len(X_external))

print("\nFiles ready for the LSTM training notebook:")
print("1.", OUTPUT_X)
print("2.", OUTPUT_Y)
print("3.", OUTPUT_METADATA)

print("\nAtlanta scalers used:")
print("X:", ATLANTA_X_SCALER_FILE)
print("y:", ATLANTA_Y_SCALER_FILE)


# Important: update the LSTM training notebook

The training notebook should load:

```python
birmingham_external_X_test.npy
birmingham_external_y_test.npy
```

instead of using:

```python
birmingham_X_test.npy
birmingham_y_test.npy
```

The Atlanta files remain unchanged:

```text
atlanta_X_train.npy
atlanta_y_train.npy
atlanta_X_val.npy
atlanta_y_val.npy
atlanta_X_test.npy
atlanta_y_test.npy
atlanta_X_scaler.pkl
atlanta_y_scaler.pkl
```

The final experimental flow is therefore:

```text
ATLANTA
Train → Validation → Internal Test
          ↓
      LSTM model
          ↓
BIRMINGHAM EXTERNAL TEST
          ↓
Cross-city metrics
```

No Birmingham-fitted scaler is used.
